# Avito: кандидатогенерация — v5, размытые локации и микрокатегории по похожим запросам

Продолжение `04_ranker_embeddings.ipynb` (тег `v4`, LB 0,8441). Использует тот же артефакт эмбеддингов из `03_embeddings.ipynb`; энкодер заново не обучается.

**Что показал разбор v4.** Полнота пула на валидации — 0,967, Recall@50 — 0,916. Самый слабый сегмент — запросы с «региональной» локацией поиска (такой локации нет ни у одного объявления, например целая область): полнота пула 0,862 и Recall@50 0,780 против 0,944 у остальных. Второй резерв — новые тексты запросов: для них P(микрокатегория | запрос) сваливается к отдельным леммам и категории поиска.

**Что меняется в v5.** Три дополнения; всё остальное — как в v4:

1. **Размытые локации.** Для каждой локации поиска по train считаются её «ядро» (города, куда уходит 95% переходов), P(перехода), нормированная на главный город, и собственный радиус (квантиль расстояний до выбранных объявлений). Для размытых локаций — регион или город, откуда больше половины переходов уходит в другие города, — три новых списка кандидатов (по тексту, по вероятным микрокатегориям, по эмбеддингам) с этой близостью. Для ранкера — признаки `loc_pn`, `loc_cover`, `dist_rel`, `q_self_share`, `q_diffuse`, `rank_region`.
2. **Микрокатегории по похожим запросам.** k ближайших по эмбеддингам текстов train (только тех, что есть в статистиках выборки) и распределение выбранных по ним микрокатегорий. Новый список `src_knn_loc` и признаки `logp_knn`, `knn_rank`, `q_knn_sim`, `q_knn_max`, `rank_knn_loc`.
3. **Шире поиск по эмбеддингам:** `src_dense_loc` 300 → 400 и новый список `src_dense` без учёта локации (150) с признаком `rank_dense`.

Выборки валидации и фолдов — ровно те же, что в v4, поэтому числа напрямую сравнимы с `report_v4.json`.

**GPU.** Нужен один раз: для п. 2 требуются вектора всех текстов запросов train, а в артефакте `03` есть только запросы бенчмарка и выборок. Ноутбук докодирует их моделью из артефакта (на A100 — несколько минут, на CPU — десятки минут) и сохраняет рядом с артефактом (`train_queries.*`). Повторные запуски берут вектора из файлов, нейросеть не запускают, и ответ не зависит от машины. Поиск соседей — точный целочисленный, как и близость векторов в v4.

| Раздел | Что происходит |
|---|---|
| 0. Окружение | код из репозитория, зависимости, сиды, артефакт эмбеддингов |
| 1. Данные | загрузка, сегменты запросов, номера текстов запросов |
| 2. Вектора запросов train | дополнение к артефакту (GPU только при первом запуске) |
| 3. Выборки | те же, что в v4 |
| 4. Пулы и выбор схемы | пулы v5, вклад новых списков, точность микрокатегорий |
| 5. Фолды для ранкера | обучающие строки |
| 6. Обучение ранкера | LightGBM, ранняя остановка по Recall@50 |
| 7. Качество на валидации | сравнение с v4, сегменты, важность признаков |
| 8. Бенчмарк | `answer.csv`, проверка формата, отличие от v4 |
| 9. Артефакты | модель, отчёт, md5 |

Полный прогон — около 1,5 часа, нужно ~16 ГБ RAM. Проверка кода: `SMOKE_TEST = True`. Запуск в JupyterLab — как у `04`, см. `JUPYTERLAB.md`.

## Настройки запуска

Единственная ячейка, которую может понадобиться поправить. Значение `None` означает «определить автоматически». Эта же ячейка помечена тегом `parameters`, поэтому при запуске через papermill любую настройку можно передать снаружи: `-p DRY_RUN True`.

In [1]:
# ── Пути (None — автоматически) ──────────────────────────────────────────────────────────
REPO_DIR = None       # папка репозитория, если ноутбук открыт не из его папки notebooks/
DATA_DIR = None       # папка с train.parquet и benchmark_*.parquet; по умолчанию <репозиторий>/data
WORK_DIR = None       # куда писать отчёт и модель ранкера; по умолчанию <репозиторий>/artifacts
OUTPUT_DIR = None     # куда писать answer.csv; по умолчанию <репозиторий>/outputs
EMB_DIR = None        # папка артефакта из 03_embeddings; по умолчанию <WORK_DIR>/embeddings

# ── Режим ─────────────────────────────────────────────────────────────────────────────────
SMOKE_TEST = False    # True — маленькие выборки (проверка кода)

## 0. Окружение

Ячейка переносит настройки в переменные окружения, находит репозиторий (поднимаясь от текущей папки), подключает его код и ставит **только недостающие** лёгкие пакеты нужных версий. Уже установленные numpy, pandas и torch не трогаются: переустановка torch в готовой GPU-среде может сломать поддержку CUDA. Число потоков BLAS фиксируется **до** импорта numpy — это часть воспроизводимости.

In [2]:
import base64, importlib, os, subprocess, sys
from pathlib import Path

# 1. Настройки → переменные окружения (их читают модули src). Уже заданные снаружи не сбрасываются.
for _name in ("REPO_DIR", "DATA_DIR", "WORK_DIR", "OUTPUT_DIR", "EMB_DIR", "HF_HOME", "HF_ENDPOINT"):
    if globals().get(_name):
        os.environ[_name] = str(globals()[_name])
for _name in ("DRY_RUN", "SMOKE_TEST"):
    if globals().get(_name):
        os.environ[_name] = "1"

# 2. Фиксированное число потоков BLAS — до импорта numpy
N_THREADS = 4
for _var in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
             "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_var] = str(N_THREADS)

# 3. Код решения
REPO_URL = "https://github.com/mishin-mikhail/avito_autumn_dev.git"
REPO_REF = "main"            # ветка, тег или хеш коммита — используется, только если репозиторий клонирует сам ноутбук


def _github_token():
    """GITHUB_TOKEN из окружения или из Kaggle Secrets (None, если его нет)."""
    if os.environ.get("GITHUB_TOKEN"):
        return os.environ["GITHUB_TOKEN"]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        return None


def _git(*args, token=None) -> str:
    """git без утечки токена: заголовок авторизации передаётся через переменные окружения."""
    env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
    if token:
        basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
        env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader",
                   GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {basic}")
    result = subprocess.run(["git", *args], env=env, capture_output=True, text=True)
    if result.returncode != 0:
        error = result.stderr.replace(token, "***") if token else result.stderr
        raise RuntimeError(f"git завершился с ошибкой:\n{error}")
    return result.stdout.strip()


def find_repo_root() -> Path:
    """REPO_DIR → папки выше текущей → (Kaggle или GITHUB_TOKEN) клонирование в текущую папку."""
    candidates = [Path(os.environ["REPO_DIR"]).expanduser()] if os.environ.get("REPO_DIR") else []
    candidates += [Path.cwd(), *Path.cwd().parents]
    for path in candidates:
        if (path / "src" / "pipeline.py").exists():
            return path.resolve()
    token = _github_token()
    if not (token or Path("/kaggle/input").exists()):
        raise RuntimeError(
            "Не найден код решения (папка src/). Откройте ноутбук из папки notebooks/ клонированного "
            "репозитория или укажите путь к репозиторию в настройке REPO_DIR.")
    target = Path("/kaggle/working/avito-candgen") if Path("/kaggle/working").exists() else Path.cwd() / "avito-candgen"
    if not target.exists():
        _git("clone", "--quiet", REPO_URL, str(target), token=token)
    _git("-C", str(target), "fetch", "--quiet", "--tags", "--force", "origin", token=token)
    is_branch = subprocess.run(["git", "-C", str(target), "rev-parse", "--verify", "--quiet",
                                f"origin/{REPO_REF}"], capture_output=True).returncode == 0
    _git("-C", str(target), "checkout", "--quiet", "--force", "--detach",
         f"origin/{REPO_REF}" if is_branch else REPO_REF)
    return target


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
try:
    COMMIT = subprocess.run(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"],
                            capture_output=True, text=True).stdout.strip() or "(не git-репозиторий)"
except FileNotFoundError:
    COMMIT = "(git не установлен)"


# 4. Недостающие пакеты. Ставятся в окружение текущего ядра; при нехватке прав — в --user.
def ensure_packages(requirements: dict) -> None:
    """requirements: модуль → pip-спецификации. Ставит только то, чего нет."""
    missing = []
    for module, specs in requirements.items():
        try:
            importlib.import_module(module)
        except ImportError:
            missing += specs
    if not missing:
        return
    print("устанавливаю:", " ".join(missing))
    cmd = [sys.executable, "-m", "pip", "install", "-q", *missing]
    if subprocess.run(cmd).returncode != 0 and subprocess.run(cmd + ["--user"]).returncode != 0:
        raise RuntimeError(f"Не удалось установить {missing}. Установите их вручную в терминале.")
    importlib.invalidate_caches()
    import site
    if site.getusersitepackages() not in sys.path:
        sys.path.append(site.getusersitepackages())


ensure_packages({
    "pyarrow": ["pyarrow"],
    "pymorphy3": ["pymorphy3==2.0.6", "pymorphy3-dicts-ru==2.4.417150.4580142"],
})

# LightGBM — ровно той версии, что в requirements.txt: от неё зависит модель ранкера, а значит и answer.csv.
from importlib import metadata
LIGHTGBM_VERSION = "4.6.0"
try:
    _lgb_installed = metadata.version("lightgbm")
except metadata.PackageNotFoundError:
    _lgb_installed = None
if _lgb_installed != LIGHTGBM_VERSION:
    if "lightgbm" in sys.modules:
        raise RuntimeError(f"В ядре уже загружен lightgbm {_lgb_installed} — перезапустите ядро (Kernel → Restart)")
    print(f"lightgbm: {_lgb_installed} → {LIGHTGBM_VERSION}")
    if subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"lightgbm=={LIGHTGBM_VERSION}"]).returncode != 0:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--user", f"lightgbm=={LIGHTGBM_VERSION}"],
                       check=True)
    importlib.invalidate_caches()
print(f"репозиторий: {REPO_ROOT}\nкоммит: {COMMIT}")

устанавливаю: pymorphy3==2.0.6 pymorphy3-dicts-ru==2.4.417150.4580142
lightgbm: 4.1.0 → 4.6.0
репозиторий: /home/jovyan/persistent_volume/avito_autumn/avito_autumn_dev
коммит: ce6fe837ad9cbc913667b726caa3c9f3494e24f6


In [3]:
import gc
import json

import numpy as np
import pandas as pd
from IPython.display import display

from src import analysis, eda
from src import encoder as enc
from src import knn_prior
from src.artifacts import find_embeddings_dir
from src.candidates import BASE_FEATURES, SOURCES, V5_FEATURES, V5_SOURCES, build_vocabs
from src.config import CFG, EMB_CFG, RANKER_V5
from src.data import load_benchmark, load_train, load_train_items_text
from src.paths import get_data_dir, get_output_dir, get_work_dir
from src.pipeline import STATS_COLS_V5, ItemLemmaCache, add_lemma_keys, build_index, build_pool
from src.ranker import (add_stage1, importance_table, ranker_features, ranker_score,
                        sample_training_rows, train_ranker)
from src.ranking import PoolData, linear_score, predict_from_scores, weights_vector
from src.repro import file_md5, library_versions, seed_everything
from src.sampling import build_folds, build_validation, group_table, scheme_keys
from src.submit import save_answer, validate_answer
from src.text import Lemmatizer
from src.utils import memory_status, resources_report, timer
from src.validation import add_query_segments, mark_seen, per_query_recall, recall_at_k
from dataclasses import replace

assert CFG.n_threads == N_THREADS
seed_everything(CFG.seed)
pd.set_option("display.max_colwidth", 100)

SMOKE = os.environ.get("SMOKE_TEST") == "1"
R = RANKER_V5          # все параметры v5 — в src/config.py
if SMOKE:
    R = replace(R, n_val_queries=300, fold_queries=300, max_rounds=150, early_stopping=30)
FEATURES = ranker_features(use_dense=True, v5=True)
DATA_DIR, WORK_DIR, OUT_DIR = get_data_dir(), get_work_dir(), get_output_dir()
VERSIONS = library_versions()
K, DEC = CFG.top_k, CFG.score_decimals
V2_W = weights_vector(BASE_FEATURES, R.v2_weights)
V4 = {"lb": 0.844087, "answer_md5": "cbd47d4544e35d176cbaa76094bc60c4"}   # отправка v4 (тег v4)
print(f"версия решения: {R.version}{' (SMOKE_TEST)' if SMOKE else ''}\ndata: {DATA_DIR}\nwork: {WORK_DIR}\nout:  {OUT_DIR}")
resources_report(WORK_DIR, need_ram_gb=16, need_disk_gb=3)
print(VERSIONS)

# --- артефакт эмбеддингов из 03_embeddings.ipynb ---
EMB_DIR = find_embeddings_dir()
EMB_MANIFEST = json.loads((EMB_DIR / "manifest.json").read_text())
print(f"\nэмбеддинги: {EMB_DIR}\n  модель {EMB_MANIFEST['model_name']} ({EMB_MANIFEST['mode']}), "
      f"объявлений {EMB_MANIFEST['n_items']:,}, запросов {EMB_MANIFEST['n_queries']:,}, "
      f"размерность {EMB_MANIFEST['dim']}, Recall@100 после дообучения {EMB_MANIFEST['recall@100_tuned']:.4f}")
if EMB_MANIFEST["mode"] != "full" and not SMOKE:
    print("[warn] артефакт собран в быстром режиме — для отправки нужен полный прогон 03_embeddings")

версия решения: v5
data: /home/jovyan/persistent_volume/avito_autumn/avito_autumn_dev/data
work: /home/jovyan/persistent_volume/avito_autumn/avito_autumn_dev/artifacts
out:  /home/jovyan/persistent_volume/avito_autumn/avito_autumn_dev/outputs
RAM: 15.5 ГБ свободно из 16.0 | диск в /home/jovyan/persistent_volume/avito_autumn/avito_autumn_dev/artifacts: 38 ГБ свободно | ядер CPU: 48
[warn] ноутбуку нужно около 16 ГБ RAM — возможна нехватка памяти
{'python': '3.10.13', 'numpy': '1.26.2', 'pandas': '2.0.3', 'scipy': '1.11.4', 'sklearn': '1.3.2', 'pyarrow': '14.0.1', 'pymorphy3': '2.0.6', 'lightgbm': '4.6.0', 'torch': '2.1.1+cu118'}

эмбеддинги: /home/jovyan/persistent_volume/avito_autumn/avito_autumn_dev/artifacts/embeddings
  модель intfloat/multilingual-e5-base (full), объявлений 208,905, запросов 25,638, размерность 768, Recall@100 после дообучения 0.3910


## 1. Данные

Как в v4, плюс номер текста запроса у каждой строки train (`qtext`): текст — это запрос вместе с фильтрами, ровно в том виде, в каком его видит энкодер. По этим номерам считается распределение микрокатегорий у соседних запросов.

In [4]:
with timer("загрузка"):
    train = load_train(DATA_DIR)
    bench_q, bench_items = load_benchmark(DATA_DIR)

lem = Lemmatizer()
with timer("подготовка запросов"):
    add_lemma_keys(train, lem)
    add_lemma_keys(bench_q, lem)
    item_locations = pd.Index(pd.concat([train["item_location_id"], bench_items["item_location_id"]]).unique())
    add_query_segments(train, item_locations)
    add_query_segments(bench_q, item_locations)
    mark_seen(bench_q, train["norm_text"].unique())
    groups = group_table(train, bench_items["item_id"])
    TRAIN_TEXTS, train["qtext"] = knn_prior.train_text_codes(train)

OVERLAP = eda.overlap(train, bench_q, bench_items, CFG.item_stats_min_overlap)
print(f"\nгрупп-запросов: {len(groups):,}; из них все выбранные объявления в корпусе: "
      f"{groups['in_corpus'].sum():,} ({groups['in_corpus'].mean():.3f})")
print(f"уникальных текстов запросов (с фильтрами) в train: {len(TRAIN_TEXTS):,}")

[загрузка] 14.6 c
[подготовка запросов] 4.1 c

── Пересечение бенчмарка с train ─────────────────────────────────────────
объявлений корпуса, встречающихся в train: 0.096
запросов, чей текст встречается в train:  0.375
запросов, чей полный ключ есть в train:   0.044
→ статистики по item_id выключены (порог 0.5)

групп-запросов: 354,241; из них все выбранные объявления в корпусе: 18,415 (0.052)
уникальных текстов запросов (с фильтрами) в train: 107,957


## 2. Вектора запросов train

Нужны для поиска похожих запросов. При первом запуске их кодирует дообученная модель из артефакта (`<EMB_DIR>/model`) — на GPU, если он есть; результат сохраняется рядом с артефактом (`train_queries.parquet`, `train_query_embeddings.npy`, `train_queries.json` с md5). При следующих запусках вектора читаются из файлов.

Для проверяющего: эти три файла — часть артефакта v5, их нужно выложить вместе с остальным артефактом эмбеддингов.

In [5]:
def encode_train_texts(texts: list) -> np.ndarray:
    """Кодирует тексты запросов моделью из артефакта (так же, как 03 кодировал запросы для 04)."""
    try:
        import torch
    except ImportError as error:
        raise RuntimeError("Векторов запросов train ещё нет, а для их расчёта нужен PyTorch: запустите "
                           "ноутбук в ядре с torch (то же, где выполнялся 03_embeddings)") from error
    info = enc.device_info()
    amp = enc.choose_amp(EMB_CFG.amp_dtype, info)
    print(f"  устройство: {info['name']}, точность: {amp.name}")
    if info["device"] == "cpu":
        print("  [warn] GPU нет — на CPU кодирование займёт десятки минут")
    model = enc.BiEncoder.load(EMB_DIR / "model", info["device"], amp)
    emb = model.encode(texts, EMB_CFG.max_len_query, EMB_CFG.encode_batch, log_every=100)
    del model
    if info["device"] == "cuda":
        torch.cuda.empty_cache()
    return emb


with timer("вектора запросов train"):
    _emb = knn_prior.train_query_embeddings(
        EMB_DIR, TRAIN_TEXTS, encode_train_texts,
        meta={"model_name": EMB_MANIFEST["model_name"], "item_artifact_md5": EMB_MANIFEST["md5"],
              "commit": COMMIT, "versions": VERSIONS})
    TEXT_INDEX = knn_prior.TextIndex(_emb)
    del _emb
TRAIN_TEXTS_MANIFEST = json.loads((EMB_DIR / knn_prior.MANIFEST_FILE).read_text())
print(f"текстов: {TEXT_INDEX.n:,}; md5 дополнения: {TRAIN_TEXTS_MANIFEST['md5']}")
memory_status("после векторов запросов train")

кодирую 107,957 текстов запросов train (есть в артефакте: 0)
  устройство: NVIDIA A100 80GB PCIe MIG 2g.20gb, точность: bf16
  закодировано 512 из 107,957
  закодировано 51,712 из 107,957
  закодировано 102,912 из 107,957
[вектора запросов train] 29.9 c
текстов: 107,957; md5 дополнения: {'train_queries.parquet': '86c91dca10d3e5312fbc02b69cb08c7e', 'train_query_embeddings.npy': '8828a2378d3b5e7e6369eb3aec3d9339'}
[память] после векторов запросов train: ноутбук занимает 5.3 ГБ, свободно 11.0 из 16.0 ГБ


## 3. Выборки валидации и фолдов

Те же функции и параметры, что в v4 (`uniform_unseen_texts=True`), поэтому валидация и фолды совпадают с v4 запрос в запрос, а числа сравнимы с `report_v4.json`. Энкодер эти запросы не видел при дообучении (`03_embeddings` исключил их строки).

In [6]:
ALL_ROWS = np.ones(len(train), dtype=bool)
KEYS = scheme_keys(groups)
with timer("выборки валидации и фолдов"):
    VAL = build_validation(train, groups, bench_q, R)
    FOLDS_ALL = {name: build_folds(train, groups, bench_q, R, VAL[name], KEYS[name]) for name in VAL}
print("запросов в фолдах:", {name: [len(f.queries) for f in folds] for name, folds in FOLDS_ALL.items()})

# Дальше из train нужны только колонки для статистик
train = train[STATS_COLS_V5].copy()
memory_status("после выборок")

pd.concat({name: s.report.set_index(["текст", "страта"])["факт"] for name, s in VAL.items()},
          axis=1).assign(цель=VAL["injected"].report["цель"].to_numpy())

[warn] валидация in_corpus: в некоторых ячейках не хватило запросов — 2274 из 2500
[warn] фолд 0: в некоторых ячейках не хватило запросов — 3220 из 4000
[warn] фолд 1: в некоторых ячейках не хватило запросов — 2867 из 4000
[warn] фолд 2: в некоторых ячейках не хватило запросов — 2548 из 4000
[warn] фолд 3: в некоторых ячейках не хватило запросов — 2088 из 4000
[выборки валидации и фолдов] 25.7 c
запросов в фолдах: {'injected': [4000, 4000, 4000, 4000], 'in_corpus': [3220, 2867, 2548, 2088]}
[память] после выборок: ноутбук занимает 5.3 ГБ, свободно 11.0 из 16.0 ГБ


injected  in_corpus  цель
текст    страта                                                           
знакомый фильтр есть | локация обычная                443        443   443
         фильтр есть | локация только поисковая        84         84    84
         фильтра нет | локация обычная                333        333   333
         фильтра нет | локация только поисковая        78         78    78
новый    фильтр есть | локация обычная                336        336   336
         фильтр есть | локация только поисковая        61         61    61
         фильтра нет | локация обычная                954        782   954
         фильтра нет | локация только поисковая       211        157   211

## 4. Пулы и выбор схемы валидации

Как в v4: индекс корпуса бенчмарка и его копии с подмешанными эталонами (`injected`), пулы для обеих схем и для бенчмарка. В `make_pool` добавлен индекс текстов train: для каждой выборки соседи ищутся только среди текстов, которые есть в её статистиках.

In [7]:
vocabs = build_vocabs([train, bench_q], [train, bench_items])
cache = ItemLemmaCache(lem, CFG.desc_max_chars)
bench_ids = frozenset(bench_items["item_id"])


def items_with_injected(samples) -> pd.DataFrame:
    """Корпус бенчмарка + эталонные объявления выборок, которых в нём нет."""
    missing = sorted(dict.fromkeys(i for s in samples for rel in s.truth for i in rel if i not in bench_ids))
    extra = load_train_items_text(DATA_DIR, missing)
    print(f"подмешано объявлений: {len(extra):,}")
    return pd.concat([bench_items, extra[bench_items.columns]], ignore_index=True)


def make_index(name, items):
    """Индекс корпуса + вектора его объявлений из артефакта."""
    return build_index(name, items, cache=cache, vocabs=vocabs, cfg=CFG, ext_cfg=R,
                       item_emb=enc.load_item_embeddings(EMB_DIR, items["item_id"]))


def make_pool(name, queries, truth, index, items, stats_mask):
    """Пул v5 с признаками и скором формулы v2 (stage1). Вектора запросов берутся из артефакта;
    докодирование — только запасной путь, как в v4 (для отправки так не делать)."""
    query_emb = enc.load_query_embeddings(EMB_DIR, enc.query_texts(queries), encode_train_texts)
    _, pool = build_pool(name, index, items, queries, train.loc[stats_mask], lem=lem,
                         vocabs=vocabs, cfg=CFG, use_item_stats=OVERLAP["use_item_stats"],
                         truth=truth, ext_cfg=R, query_emb=query_emb, text_index=TEXT_INDEX)
    return add_stage1(pool, index, R.v2_weights, DEC)


bench_index = make_index("корпус бенчмарка", bench_items)
inj_items = items_with_injected([VAL["injected"], *FOLDS_ALL["injected"]])
inj_index = make_index("корпус injected", inj_items)
CORPORA = {"injected": (inj_index, inj_items), "in_corpus": (bench_index, bench_items)}

VAL_POOLS = {name: make_pool(f"валидация {name}", s.queries, s.truth, *CORPORA[name], s.stats_mask)
             for name, s in VAL.items()}
BENCH_POOL = make_pool("бенчмарк", bench_q, None, bench_index, bench_items, ALL_ROWS)
cache.clear()
memory_status("после пулов валидации и бенчмарка")

[корпус бенчмарка: индекс корпуса] 139.5 c
подмешано объявлений: 19,693
[корпус injected: индекс корпуса] 59.0 c
  пул: 64/2500 запросов
  пул: 384/2500 запросов
  пул: 704/2500 запросов
  пул: 1024/2500 запросов
  пул: 1344/2500 запросов
  пул: 1664/2500 запросов
  пул: 1984/2500 запросов
  пул: 2304/2500 запросов
[валидация injected: статистики и пул] 131.3 c
валидация injected: запросов 2,500, строк пула 3,291,569 (~1317 на запрос)
  пул: 64/2274 запросов
  пул: 384/2274 запросов
  пул: 704/2274 запросов
  пул: 1024/2274 запросов
  пул: 1344/2274 запросов
  пул: 1664/2274 запросов
  пул: 1984/2274 запросов
  пул: 2274/2274 запросов
[валидация in_corpus: статистики и пул] 103.4 c
валидация in_corpus: запросов 2,274, строк пула 3,040,841 (~1337 на запрос)
  пул: 64/2452 запросов
  пул: 384/2452 запросов
  пул: 704/2452 запросов
  пул: 1024/2452 запросов
  пул: 1344/2452 запросов
  пул: 1664/2452 запросов
  пул: 1984/2452 запросов
  пул: 2304/2452 запросов
[бенчмарк: статистики и пул] 

**Калибровка** — как в v4: Recall@50 отправленной модели v2 (пул v2 + веса v2) на каждой схеме сравнивается с её результатом на лидерборде; выбирается схема с наименьшим расхождением.

In [8]:
calibration = []
for name, s in VAL.items():
    pool, (index, _) = VAL_POOLS[name], CORPORA[name]
    v2_pool = analysis.v2_rows(pool)
    recall_v2 = PoolData(v2_pool, index, BASE_FEATURES, s.n_rel).recall(V2_W, K, DEC)
    calibration.append({
        "схема": name, "запросов": len(s.queries), "Recall@50 v2": recall_v2,
        "LB v2": R.v2_lb, "расхождение": recall_v2 - R.v2_lb,
        "полнота пула v2": analysis.pool_hit_rate(v2_pool, s.n_rel).mean(),
        "полнота пула v5": analysis.pool_hit_rate(pool, s.n_rel).mean(),
    })
calibration = pd.DataFrame(calibration).set_index("схема").round(4)
SCHEME = calibration["расхождение"].abs().idxmin() if R.val_scheme == "auto" else R.val_scheme
display(calibration)
print(f"→ схема валидации: {SCHEME}")

,запросов,Recall@50 v2,LB v2,расхождение,полнота пула v2,полнота пула v5
схема,,,,,,
injected,2500,0.8597,0.8313,0.0284,0.9535,0.9828
in_corpus,2274,0.8761,0.8313,0.0448,0.9675,0.9908


→ схема валидации: injected


**Вклад новых списков.** Полнота каждого списка отдельно, всего пула и пула v4 (без списков v5; у `src_dense_loc` в v4 было 300 кандидатов, здесь 400, поэтому «пул v4» — оценка сверху). Ниже — то же по сегментам: главный вопрос v5 — выросла ли полнота у запросов с региональной локацией и у новых текстов.

In [9]:
V4_SOURCES = SOURCES + ["src_char_loc", "src_dense_loc"]


def pool_v4_rows(pool):
    return pool[pool[V4_SOURCES].any(axis=1)]


for name, s_ in VAL.items():
    pool = VAL_POOLS[name]
    table = analysis.source_recall(pool, s_.n_rel)
    table.loc["пул v4 (без списков v5)", "recall"] = analysis.pool_hit_rate(pool_v4_rows(pool), s_.n_rel).mean()
    for src in V5_SOURCES:        # что теряется, если убрать один список v5
        rest = [c for c in V4_SOURCES + V5_SOURCES if c != src]
        table.loc[f"пул без {src}", "recall"] = analysis.pool_hit_rate(pool[pool[rest].any(axis=1)], s_.n_rel).mean()
    print(f"\nвалидация {name}:")
    display(table.round(4))

s_, pool = VAL[SCHEME], VAL_POOLS[SCHEME]
seg = s_.queries[["seg_text", "seg_loc"]].assign(
    q_diffuse=pool.groupby("q")["q_diffuse"].first().reindex(range(len(s_.queries))).fillna(0).to_numpy() > 0,
    **{"пул v4": analysis.pool_hit_rate(pool_v4_rows(pool), s_.n_rel),
       "пул v5": analysis.pool_hit_rate(pool, s_.n_rel)})
print(f"\nполнота пула по сегментам, валидация {SCHEME}:")
display(pd.concat({col: seg.groupby(col)[["пул v4", "пул v5"]].mean().assign(запросов=seg.groupby(col).size())
                   for col in ["seg_text", "seg_loc", "q_diffuse"]}).round(4))


валидация injected:


,recall,avg_candidates
pool,0.9828,1316.6276
src_char_loc,0.7752,1316.6276
src_dense,0.5122,1316.6276
src_dense_loc,0.8341,1316.6276
src_dense_region,0.1550,1316.6276
src_knn_loc,0.8648,1316.6276
src_memo,0.0000,1316.6276
src_prior_loc,0.8209,1316.6276
src_prior_region,0.1109,1316.6276
src_text,0.6526,1316.6276



валидация in_corpus:


,recall,avg_candidates
pool,0.9908,1337.2212
src_char_loc,0.7842,1337.2212
src_dense,0.5390,1337.2212
src_dense_loc,0.8477,1337.2212
src_dense_region,0.1480,1337.2212
src_knn_loc,0.8865,1337.2212
src_memo,0.0000,1337.2212
src_prior_loc,0.8377,1337.2212
src_prior_region,0.1117,1337.2212
src_text,0.6961,1337.2212



полнота пула по сегментам, валидация injected:


пул v4  пул v5  запросов
seg_text  знакомый                  0.9779  0.9876       938
          новый                     0.9609  0.9798      1562
seg_loc   локация обычная           0.9890  0.9909      2066
          локация только поисковая  0.8641  0.9439       434
q_diffuse False                     0.9888  0.9908      2038
          True                      0.8723  0.9473       462

**Точность микрокатегорий.** Для запросов, у которых эталон попал в пул: на каком месте микрокатегория эталона в P(микрокатегория | запрос) по леммам (`prior_rank`, как в v2–v4) и по соседним запросам (`knn_rank`, v5). Главное — строка «новый»: для новых текстов P по леммам слабее всего.

In [10]:
pos = pool[pool["label"] == 1]
best = pos.groupby("q")[["prior_rank", "knn_rank"]].min()
best["seg_text"] = s_.queries["seg_text"].to_numpy()[best.index]
micro_table = best.groupby("seg_text").agg(
    запросов=("prior_rank", "size"),
    top1_леммы=("prior_rank", lambda r: (r < 1).mean()), top1_соседи=("knn_rank", lambda r: (r < 1).mean()),
    top3_леммы=("prior_rank", lambda r: (r < 3).mean()), top3_соседи=("knn_rank", lambda r: (r < 3).mean()))
display(micro_table.round(4))
knn_sim = pool.groupby("q")["q_knn_sim"].first()
print("близость лучшего соседа, медиана по сегментам:",
      knn_sim.groupby(s_.queries["seg_text"].to_numpy()[knn_sim.index]).median().round(4).to_dict())

,запросов,top1_леммы,top1_соседи,top3_леммы,top3_соседи
seg_text,,,,,
знакомый,928,0.8050,0.8233,0.9256,0.9289
новый,1531,0.4722,0.6675,0.6799,0.8406


близость лучшего соседа, медиана по сегментам: {'знакомый': 0.9998999834060669, 'новый': 0.9666000008583069}


In [11]:
# Проверка: пайплайн без новых кандидатов и с весами v2 даёт ровно отправленный ответ v2
v2_bench = analysis.v2_rows(BENCH_POOL)
v2_pred = predict_from_scores(v2_bench, bench_index, linear_score(v2_bench[BASE_FEATURES].to_numpy(np.float64), V2_W),
                              K, DEC, len(bench_q))
v2_md5 = save_answer(bench_q["query_id"], v2_pred, WORK_DIR / "answer_v2_check.csv")
print(f"md5 ответа v2 из пайплайна v5: {v2_md5} | отправленный: {R.v2_answer_md5} | "
      f"совпадает: {v2_md5 == R.v2_answer_md5}")

md5 ответа v2 из пайплайна v5: 2de61da58afd87fc244e225b7cccde58 | отправленный: 2de61da58afd87fc244e225b7cccde58 | совпадает: True


## 5. Фолды для ранкера

Как в v4: фолды построены вместе с валидацией, статистики каждого фолда (в том числе соседи для P(микрокатегория) и переходы локаций) считаются без его собственных строк. Первые `n_folds − 1` фолдов — обучение, последний — ранняя остановка.

In [12]:
FOLDS = FOLDS_ALL[SCHEME]
fold_index, fold_items = CORPORA[SCHEME]

# Освобождаем память: пулы, выборки и индекс другой схемы больше не нужны
for name in [n for n in VAL_POOLS if n != SCHEME]:
    del VAL_POOLS[name]
if SCHEME == "in_corpus":
    del CORPORA["injected"], inj_index, inj_items
del FOLDS_ALL, pool, pos
memory_status("после выбора схемы")

[память] после выбора схемы: ноутбук занимает 12.5 ГБ, свободно 3.8 из 16.0 ГБ


In [13]:
TRAIN_COLS = ["gid", "label"] + FEATURES
VALID_COLS = ["q", "item", "label"] + FEATURES
train_parts = []
for f, fold in enumerate(FOLDS):
    pool = make_pool(f"фолд {f}", fold.queries, fold.truth, fold_index, fold_items, fold.stats_mask)
    if f < R.n_folds - 1:
        rows = sample_training_rows(pool, R, seed=CFG.seed + f)
        rows.insert(0, "gid", f * 1_000_000 + rows["q"].astype(np.int64))
        train_parts.append(rows[TRAIN_COLS])
        del rows
    else:
        VALID_POOL = pool[VALID_COLS]
    del pool
    gc.collect()

TRAIN_ROWS = pd.concat(train_parts, ignore_index=True)
del train_parts
print(f"обучающих строк: {len(TRAIN_ROWS):,} (позитивов {int(TRAIN_ROWS['label'].sum()):,}); "
      f"строк в фолде остановки: {len(VALID_POOL):,}")
memory_status("перед обучением ранкера")2

  пул: 64/4000 запросов
  пул: 384/4000 запросов
  пул: 704/4000 запросов
  пул: 1024/4000 запросов
  пул: 1344/4000 запросов
  пул: 1664/4000 запросов
  пул: 1984/4000 запросов
  пул: 2304/4000 запросов
  пул: 2624/4000 запросов
  пул: 2944/4000 запросов
  пул: 3264/4000 запросов
  пул: 3584/4000 запросов
  пул: 3904/4000 запросов
[фолд 0: статистики и пул] 205.5 c
фолд 0: запросов 4,000, строк пула 5,290,412 (~1323 на запрос)
  пул: 64/4000 запросов
  пул: 384/4000 запросов
  пул: 704/4000 запросов
  пул: 1024/4000 запросов
  пул: 1344/4000 запросов
  пул: 1664/4000 запросов
  пул: 1984/4000 запросов
  пул: 2304/4000 запросов
  пул: 2624/4000 запросов
  пул: 2944/4000 запросов
  пул: 3264/4000 запросов
  пул: 3584/4000 запросов
  пул: 3904/4000 запросов
[фолд 1: статистики и пул] 194.1 c
фолд 1: запросов 4,000, строк пула 5,296,479 (~1324 на запрос)
  пул: 64/4000 запросов
  пул: 384/4000 запросов
  пул: 704/4000 запросов
  пул: 1024/4000 запросов
  пул: 1344/4000 запросов
  пул: 166

## 6. Обучение ранкера

Как в v4: `lambdarank` и `binary`, ранняя остановка и выбор — по Recall@50 на полном пуле последнего фолда. Признаков стало на 12 больше.

In [14]:
valid_fold = FOLDS[-1]
valid_rank = fold_index.rank[VALID_POOL["item"].to_numpy()]
fold_scores = {"формула v2": PoolData(VALID_POOL, fold_index, BASE_FEATURES, valid_fold.n_rel).recall(V2_W, K, DEC)}
print(f"признаков у ранкера: {len(FEATURES)} (из них новых в v5: {len(V5_FEATURES)})")
MODELS = {}
for objective in R.objectives:
    with timer(f"LightGBM {objective}"):
        booster, best_iter, best = train_ranker(TRAIN_ROWS, VALID_POOL, valid_fold.n_rel, valid_rank,
                                                objective, R, CFG.seed, N_THREADS, K, DEC, features=FEATURES)
    MODELS[objective] = booster
    fold_scores[f"ранкер {objective}"] = best
    print(f"{objective}: лучшая итерация {best_iter}, Recall@{K} на фолде {best:.4f}")

OBJECTIVE = max(R.objectives, key=lambda o: fold_scores[f"ранкер {o}"])
RANKER = MODELS[OBJECTIVE]
pd.Series(fold_scores, name=f"Recall@{K} на фолде остановки").round(4).to_frame()

признаков у ранкера: 55 (из них новых в v5: 12)
[100]	fold's recall@50: 0.921348
[200]	fold's recall@50: 0.922169
[LightGBM lambdarank] 205.9 c
lambdarank: лучшая итерация 172, Recall@50 на фолде 0.9229
[100]	fold's recall@50: 0.920269
[200]	fold's recall@50: 0.919543
[LightGBM binary] 165.5 c
binary: лучшая итерация 149, Recall@50 на фолде 0.9218


,Recall@50 на фолде остановки
формула v2,0.8546
ранкер lambdarank,0.9229
ранкер binary,0.9218


## 7. Качество на валидации

Валидация не участвовала ни в обучении, ни в ранней остановке. Для сравнения — числа v4 на тех же запросах из `report_v4.json` (если он лежит в папке артефактов).

In [15]:
val_s, val_pool = VAL[SCHEME], VAL_POOLS[SCHEME]
val_index = CORPORA[SCHEME][0]
nq = len(val_s.queries)


def val_recall(pool, score):
    return recall_at_k(predict_from_scores(pool, val_index, score, K, DEC, nq), val_s.truth, K)


ranker_val_score = ranker_score(RANKER, val_pool, N_THREADS, FEATURES)
VAL_RESULTS = {
    "формула v2 на пуле v5": val_recall(val_pool, val_pool["stage1"].to_numpy(np.float64)),
    f"ранкер ({OBJECTIVE})": val_recall(val_pool, ranker_val_score),
}
USE_RANKER = VAL_RESULTS[f"ранкер ({OBJECTIVE})"] > VAL_RESULTS["формула v2 на пуле v5"]
POOL_RECALL = float(analysis.pool_hit_rate(val_pool, val_s.n_rel).mean())

compare = pd.DataFrame({"v5": {"полнота пула": POOL_RECALL, "Recall@50": max(VAL_RESULTS.values())}})
report_v4_path = WORK_DIR / "report_v4.json"
if report_v4_path.is_file():
    report_v4 = json.loads(report_v4_path.read_text())
    if report_v4.get("val_scheme") == SCHEME:
        compare["v4"] = {"полнота пула": report_v4["pool_recall_val"],
                         "Recall@50": max(report_v4["val_recall@50"].values())}
        compare["прирост"] = compare["v5"] - compare["v4"]
    else:
        print(f"[warn] в report_v4.json другая схема валидации ({report_v4.get('val_scheme')}) — не сравниваю")
display(pd.Series(VAL_RESULTS, name=f"Recall@{K}, валидация {SCHEME}").round(4).to_frame())
display(compare.round(4))
print(f"в ответ идёт: {'ранкер' if USE_RANKER else 'формула v2'}")

,"Recall@50, валидация injected"
формула v2 на пуле v5,0.8638
ранкер (lambdarank),0.9154


,v5,v4,прирост
Recall@50,0.9154,0.9157,-0.0002
полнота пула,0.9828,0.9669,0.0159


в ответ идёт: ранкер


In [16]:
final_val_score = ranker_val_score if USE_RANKER else val_pool["stage1"].to_numpy(np.float64)
val_pred = predict_from_scores(val_pool, val_index, final_val_score, K, DEC, nq)
r50 = per_query_recall(val_pred, val_s.truth, K)
val_queries = val_s.queries.assign(
    q_diffuse=np.where(val_pool.groupby("q")["q_diffuse"].first().reindex(range(nq)).fillna(0).to_numpy() > 0,
                       "размытая", "обычная"))
analysis.segment_table(val_queries, r50, analysis.pool_hit_rate(val_pool, val_s.n_rel),
                       seg_cols=("seg_text", "seg_filter", "seg_loc", "q_diffuse"))

n   share    pool  recall50
ось        сегмент                                                 
seg_text   знакомый                   938  0.3752  0.9876    0.9331
           новый                     1562  0.6248  0.9798    0.9048
seg_filter фильтр есть                924  0.3696  0.9848    0.9193
           фильтра нет               1576  0.6304  0.9816    0.9132
seg_loc    локация обычная           2066  0.8264  0.9909    0.9379
           локация только поисковая   434  0.1736  0.9439    0.8086
q_diffuse  обычная                   2038  0.8152  0.9908    0.9370
           размытая                   462  0.1848  0.9473    0.8202

In [17]:
importance = importance_table(RANKER)
display(importance.head(25))
print(f"доля важности новых признаков v5: {importance.loc[importance['признак'].isin(V5_FEATURES), 'доля'].sum():.3f}")
errors = analysis.error_examples(val_pred, val_s.truth, val_s.queries, CORPORA[SCHEME][1],
                                 analysis.pool_hit_rate(val_pool, val_s.n_rel), n=15)
print(f"запросов без единого попадания: {int((r50 == 0).sum())} ({(r50 == 0).mean():.3f})")
errors

,признак,gain,доля
0,stage1_rank,722431.975736,0.406809
1,rank_text_loc,318382.017952,0.179284
2,dense,185625.580019,0.104528
3,rank_dense_loc,107094.690646,0.060306
4,dist_rel,101134.510895,0.056950
5,desc,56353.558976,0.031733
6,rank_dense,47611.520495,0.026811
7,rank_knn_loc,27440.223112,0.015452
8,logp_knn,25191.102357,0.014185
9,rank_region,22623.999943,0.012740


доля важности новых признаков v5: 0.141
запросов без единого попадания: 199 (0.080)


,запрос,фильтры,та же локация,в пуле,заголовок эталона,параметры эталона
0,ремонт фундамента,Вид услуги Строительство,False,True,"Замена венцов, подъем домов, ремонт фундаментов, п","Вид услуги Строительство Место оказания услуг Ленинградская область, Гатчина, Центр Тип стоимост..."
1,массаж,"Тип услуги СПА-услуги, массаж Вид услуги Красота, здоровье",False,True,Классический расслабляющий массаж всего тела,"Вид услуги Красота, здоровье Место оказания услуг Республика Татарстан, Казань, улица Юлиуса Фуч..."
2,похудение,"Онлайн-запись Вид услуги Красота, здоровье",True,True,LPG массаж. Антицеллюлитный массаж,"Вид услуги Красота, здоровье Место оказания услуг Свердловская область, Екатеринбург, проспект К..."
3,фотомодель,Вид услуги Фото- и видеосъёмка,False,True,Модель на съемки,"Вид услуги Фото- и видеосъёмка Место оказания услуг Москва, Братиславская улица Тип стоимости за..."
4,air touch,"Тип услуги Услуги парикмахера Вид услуги Красота, здоровье",False,True,Мелирование кератин аиртач стрижка,"Вид услуги Красота, здоровье Место оказания услуг Тюмень, улица Ленина, 81 Тип услуги Услуги пар..."
5,заливка бетона,,False,True,Фундамент Отмостка Парковка,"Вид услуги Строительство Место оказания услуг Нижний Новгород, Приокский район, Цветочная улица,..."
6,ремонт фанкойлов,,False,True,"Установка и Продажа Кондиционеров,Ремонт,Заправка","Вид услуги Монтаж и установка техники Место оказания услуг Санкт-Петербург, улица Дыбенко, 26 Ти..."
7,ремонт компьютеров,Вид услуги Компьютерная помощь,True,True,Ком Тех,"Вид услуги Деловые услуги Тип услуги Место оказания услуг Краснодарский край, станица Выселки Ти..."
8,прокат мото,,False,False,Прокат снегоходов / квадроциклов / эндуро,"Вид услуги Праздники, мероприятия Место оказания услуг Республика Башкортостан, Абзелиловский ра..."
9,пескоструйная обработка рамы авто,,True,True,"Пескоструйная обработка, рам, арок","Вид услуги Оборудование, производство Тип услуги Производство, обработка Место оказания услуг ра..."


## 8. Предсказание для бенчмарка

Пул бенчмарка построен в разделе 4 по статистикам всего train. Если в папке вывода лежит `answer.csv` отправки v4 (узнаётся по md5), ноутбук показывает, насколько новый ответ от него отличается.

In [18]:
bench_score = (ranker_score(RANKER, BENCH_POOL, N_THREADS, FEATURES) if USE_RANKER
               else BENCH_POOL["stage1"].to_numpy(np.float64))
bench_pred = predict_from_scores(BENCH_POOL, bench_index, bench_score, K, DEC, len(bench_q))

ANSWER_PATH = OUT_DIR / "answer.csv"
v4_pred = None
if ANSWER_PATH.is_file() and file_md5(ANSWER_PATH) == V4["answer_md5"]:
    v4_answer = pd.read_csv(ANSWER_PATH, dtype=str, keep_default_na=False).set_index("query_id")
    v4_pred = [a.split() for a in v4_answer.loc[bench_q["query_id"].astype(str), "answer"]]
    (OUT_DIR / "answer_v4.csv").write_bytes(ANSWER_PATH.read_bytes())     # ответ v4 сохраняется рядом
save_answer(bench_q["query_id"], bench_pred, ANSWER_PATH)
CHECK = validate_answer(ANSWER_PATH, bench_q["query_id"], bench_items["item_id"], K)
print(CHECK)
overlap_v2 = np.mean([len(frozenset(a) & frozenset(b)) / K for a, b in zip(bench_pred, v2_pred)])
print(f"совпадение с ответом v2: {overlap_v2:.3f} объявлений из топ-50 в среднем")
if v4_pred is not None:
    overlap_v4 = np.mean([len(frozenset(a) & frozenset(b)) / K for a, b in zip(bench_pred, v4_pred)])
    print(f"совпадение с ответом v4: {overlap_v4:.3f} (ответ v4 сохранён как answer_v4.csv)")

{'rows': 2452, 'min_items': 50, 'max_items': 50, 'md5': 'bf25d852bce82b30561275a4b987797d'}
совпадение с ответом v2: 0.487 объявлений из топ-50 в среднем
совпадение с ответом v4: 0.768 (ответ v4 сохранён как answer_v4.csv)


## 9. Артефакты

Модель ранкера (`ranker_v5.txt`) и отчёт (`report_v5.json`) — в папке артефактов. Для воспроизведения у проверяющего нужен артефакт эмбеддингов **вместе с дополнением** `train_queries.*` — его md5 записан в отчёт. Повторный запуск должен дать тот же md5 `answer.csv`.

In [19]:
RANKER.save_model(str(WORK_DIR / f"ranker_{R.version}.txt"), num_iteration=RANKER.best_iteration)
report = {
    "version": R.version,
    "answer_md5": CHECK["md5"],
    "val_scheme": SCHEME,
    "calibration": calibration.reset_index().to_dict(orient="records"),
    "v2_reproduced": v2_md5 == R.v2_answer_md5,
    "val_recall@50": VAL_RESULTS,
    "fold_recall@50": fold_scores,
    "objective": OBJECTIVE,
    "best_iteration": RANKER.best_iteration,
    "use_ranker": bool(USE_RANKER),
    "pool_recall_val": POOL_RECALL,
    "micro_accuracy": micro_table.reset_index().to_dict(orient="records"),
    "config": CFG.as_dict(),
    "ranker_config": R.as_dict(),
    "embeddings": {k: v for k, v in EMB_MANIFEST.items() if k != "emb_config"},
    "train_queries": {k: TRAIN_TEXTS_MANIFEST[k] for k in ("n_texts", "md5")},
    "commit": COMMIT,
    "versions": VERSIONS,
}
(WORK_DIR / f"report_{R.version}.json").write_text(json.dumps(report, ensure_ascii=False, indent=1, default=str))

for name, value in VAL_RESULTS.items():
    print(f"{name:28s} {value:.4f}")
print(f"полнота пула: {POOL_RECALL:.4f}")
print(f"answer.csv md5: {CHECK['md5']}")

формула v2 на пуле v5        0.8638
ранкер (lambdarank)          0.9154
полнота пула: 0.9828
answer.csv md5: bf25d852bce82b30561275a4b987797d
